# WaterOnt GraphRAG on Colab

This notebook builds the WaterOnt knowledge graph for L-TOWN, then gives you two demonstrators:

1. **A static web page** (one HTML file) you can open anywhere or host for free on GitHub Pages.
2. **A live app** running the real Python engine, with a temporary public link from Gradio.

Run the cells in order. Asset records are synthetic; network, sensors and demands come from the L-TOWN model.

## 1. Install and load the code
Downloads the code from GitHub. Clones `github.com/manuelhf/WaterOnt`.

In [ ]:
GITHUB_USER = "manuelhf"
!pip -q install wntr rdflib gradio
!git clone -q https://github.com/{GITHUB_USER}/WaterOnt.git /content/WaterOnt
import sys, os
os.chdir("/content/WaterOnt")
sys.path.insert(0, "/content/WaterOnt")
from google.colab import files

## 2. Build the knowledge graph
About 10 seconds. Change `seed` to draw a different synthetic utility on the same network.

In [ ]:
from wateront.config import Config
from wateront.pipeline import build_knowledge_graph

cfg = Config(seed=42)
res = build_knowledge_graph("data/networks/L-TOWN.inp", cfg)
res.kg.stats()["labels"]

Ask a question directly in Python:

In [ ]:
from wateront.graphrag import GraphRAG
rag = GraphRAG(res.kg, cfg, use_llm=False)
print(rag.ask("Which unmonitored hydraulically critical mains have intolerable risk? top 5").text)

## 3. Web demonstrator
Two options. Write a complete page with this network inside (open it anywhere, or replace `demo/index.html` on GitHub), or write just a network file and open it on the hosted explorer with **Open network…**. The file is read in the browser and never uploaded.

In [ ]:
from wateront.web_export import write_web, write_json
page = write_web(res, "demo/index.html", cfg)                      # complete page
net = write_json(res, f"{res.network_id}_seed{cfg.seed}.json", cfg)  # file for "Open network…"
files.download(net)

## 4. Live app (real Python engine)
Optional: to let Claude plan queries and write answers, add your Anthropic API key in Colab's **Secrets** panel (key icon on the left) as `ANTHROPIC_API_KEY`. Without it the app uses the deterministic template generator.

The `*.gradio.live` link printed below works for about 72 hours while this notebook is running.

In [ ]:
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key found: Claude mode available")
except Exception:
    print("No API key: template mode only")

from wateront.gradio_app import build_app
app = build_app(res)
app.launch(share=True)

## 5. Experiments for the paper
Runs every result in Sections 5 and 6: the 2 x 2 comparison (with or without the knowledge graph, with or without an LLM), the planner analysis, robustness and sensitivity. The two LLM cells run only if `ANTHROPIC_API_KEY` is set in Colab Secrets (see section 4); allow about an hour.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    print("No API key found: the two LLM cells will be skipped")
!pip -q install anthropic
!python paper/experiments.py

In [ ]:
import pandas as pd
pd.read_csv("paper/results/comparison_summary.csv").round(3)   # rows for Table 5

## 6. Run the tests (optional)

In [ ]:
!pip -q install pytest
!python -m pytest -q tests